# TrustRAG: A Self-Verifying RAG System with Hallucination Detection

This notebook builds a Retrieval-Augmented Generation (RAG) pipeline on real legal contracts (CUAD dataset), then adds a verification layer that checks whether each generated answer is actually supported by the retrieved text.

**Run cells top to bottom, in order.** Each cell builds on the one before it.

Everything here uses free, open-source models — no API keys, no paid services.

## Step 1: Install Libraries

We pin `datasets==2.19.0` because newer versions dropped support for the CUAD loading script.

In [ ]:
!pip install -q "datasets==2.19.0" sentence-transformers faiss-cpu transformers accelerate


## Step 2: Load the Dataset

CUAD (Contract Understanding Atticus Dataset) — real legal contracts labeled by lawyers. Free, CC BY 4.0 licensed.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("theatticusproject/cuad-qa", trust_remote_code=True)
print(dataset)


In [ ]:
# Convert to pandas once, up front -- makes filtering by contract MUCH faster later
import pandas as pd

df_all = dataset["train"].to_pandas()
print("Total rows:", len(df_all))
print("Unique contracts:", df_all["title"].nunique())


## Step 3: Chunking Function

Splits a long document into overlapping ~500-character pieces so we can search through it.

In [ ]:
def split_into_chunks(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


## Step 4: Load the Embedding Model

Converts text chunks into numeric vectors representing their meaning.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded.")


In [ ]:
def build_index(chunks):
    """Embed a list of chunks and build a searchable FAISS index."""
    embeddings = embed_model.encode(chunks)
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(np.array(embeddings))
    return index


## Step 5: Load the Answer-Generation Model

We load the tokenizer and model directly (not via `pipeline()`), since `pipeline("text2text-generation", ...)` is not recognized by all `transformers` versions and throws a `KeyError`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

gen_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
gen_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
print("Generation model loaded.")


In [ ]:
def generate_answer(question, context_text):
    prompt = f"""Answer the question based only on the context below.

Context:
{context_text}

Question: {question}

Answer:"""
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = gen_model.generate(**inputs, max_length=100)
    return gen_tokenizer.decode(outputs[0], skip_special_tokens=True)


## Step 6: Load the Hallucination Checker (NLI Model)

This model checks whether a chunk of text *entails* (supports), *contradicts*, or is *neutral* toward the generated answer.

In [ ]:
from transformers import AutoModelForSequenceClassification

nli_tokenizer = AutoTokenizer.from_pretrained("roberta-large-mnli")
nli_model = AutoModelForSequenceClassification.from_pretrained("roberta-large-mnli")
print("NLI verification model loaded.")


In [ ]:
def check_support(context, answer):
    """Returns (verdict, confidence) -- verdict is contradiction / neutral / entailment."""
    inputs = nli_tokenizer(context, answer, return_tensors="pt", truncation=True, max_length=512)
    outputs = nli_model(**inputs)
    probs = outputs.logits.softmax(dim=1)
    labels = ["contradiction", "neutral", "entailment"]
    predicted = labels[probs.argmax().item()]
    confidence = probs.max().item()
    return predicted, confidence


## Step 7: The Full TrustRAG Pipeline

This ties everything together: retrieve relevant chunks → generate an answer → check each chunk individually for support → return the strongest verdict.

**Important fix**: we check the answer against each retrieved chunk *individually* rather than all combined together. Combining chunks into one big blob confused the NLI model and produced false "contradiction" results during testing.

In [ ]:
def trustrag_pipeline(question, chunks, index, k=3):
    # Retrieve top-k chunks
    query_embedding = embed_model.encode([question])
    distances, indices = index.search(np.array(query_embedding), k)
    retrieved_chunks = [chunks[idx] for idx in indices[0]]

    # Generate an answer using all retrieved chunks combined
    context_text = "\n".join(retrieved_chunks)
    answer = generate_answer(question, context_text)

    # Verify the answer against EACH chunk individually, keep the strongest support found
    best_verdict = "unsupported"
    best_confidence = 0.0
    supporting_chunk = None
    for chunk_text in retrieved_chunks:
        verdict, confidence = check_support(chunk_text, answer)
        if verdict == "entailment" and confidence > best_confidence:
            best_verdict = "supported"
            best_confidence = confidence
            supporting_chunk = chunk_text

    return {
        "question": question,
        "answer": answer,
        "verdict": best_verdict,
        "confidence": round(best_confidence, 2),
        "supporting_chunk": supporting_chunk,
    }


## Step 8: Quick Test on One Contract

Let's confirm everything works end-to-end before running the full evaluation.

In [ ]:
sample_title = df_all["title"].unique()[0]
sample_context = df_all[df_all["title"] == sample_title].iloc[0]["context"]

sample_chunks = split_into_chunks(sample_context)
sample_index = build_index(sample_chunks)

print(f"Contract: {sample_title}")
print(f"Number of chunks: {len(sample_chunks)}")

test_result = trustrag_pipeline("Who are the parties to this agreement?", sample_chunks, sample_index)
print("\nQuestion:", test_result["question"])
print("Answer:", test_result["answer"])
print("Verdict:", test_result["verdict"], f"(confidence: {test_result['confidence']})")


## Step 9: Stress-Test — Confirm the Checker Catches Wrong Answers

Before trusting the evaluation results, let's prove the checker actually rejects a deliberately false answer.

In [ ]:
fake_answer = "The agreement was signed by Microsoft Corporation in the year 2050."

print("Testing whether the checker catches a WRONG answer:\n")
query_embedding = embed_model.encode(["Who are the parties to this agreement?"])
distances, indices = sample_index.search(np.array(query_embedding), 3)

caught = False
for idx in indices[0]:
    chunk_text = sample_chunks[idx]
    verdict, confidence = check_support(chunk_text, fake_answer)
    print(f"Chunk #{idx}: {verdict} (confidence: {confidence:.2f})")
    if verdict == "entailment":
        caught = True

print("\nResult:", "FAILED - fake answer was wrongly supported" if caught else "PASSED - no chunk supported the fake answer")


## Step 10: Evaluate on Multiple Contracts

This builds the evaluation table and hallucination rate you need for your report. We deduplicate questions per contract, since CUAD sometimes repeats the same question across multiple annotation rows.

In [ ]:
def evaluate_contract(contract_title, chunk_size=500, overlap=50, num_questions=5):
    """Runs the full pipeline on up to num_questions unique questions for one contract."""
    contract_rows = df_all[df_all["title"] == contract_title].drop_duplicates(subset="question")
    contract_rows = contract_rows.head(num_questions)

    context_text = contract_rows.iloc[0]["context"]
    chunks = split_into_chunks(context_text, chunk_size, overlap)
    index = build_index(chunks)

    results = []
    for i, row in enumerate(contract_rows.itertuples()):
        print(f"  [{contract_title[:40]}] question {i+1}/{len(contract_rows)}...")
        result = trustrag_pipeline(row.question, chunks, index)
        results.append({
            "contract": contract_title,
            "question": result["question"],
            "generated_answer": result["answer"],
            "verdict": result["verdict"],
            "confidence": result["confidence"],
        })
    return results


In [ ]:
# Pick 3 different contracts to evaluate on
all_titles = df_all["title"].unique().tolist()
contracts_to_test = all_titles[:3]
print("Testing on these contracts:")
for t in contracts_to_test:
    print(" -", t)


In [ ]:
import time

all_results = []
start = time.time()
for title in contracts_to_test:
    all_results.extend(evaluate_contract(title, num_questions=5))
print(f"\nDone in {time.time()-start:.1f} seconds")

results_df = pd.DataFrame(all_results)
pd.set_option("display.max_colwidth", 60)
print(results_df[["contract", "question", "generated_answer", "verdict", "confidence"]].to_string())


In [ ]:
supported = (results_df["verdict"] == "supported").sum()
total = len(results_df)
print(f"SUPPORTED:   {supported}/{total} ({supported/total*100:.1f}%)")
print(f"UNSUPPORTED: {total-supported}/{total} ({(total-supported)/total*100:.1f}%)")


## Step 11 (Optional): Compare Different Chunk Sizes

Tests whether chunk size affects the hallucination rate — a useful finding for your report.

In [ ]:
chunk_size_results = {}

for size in [300, 500, 800]:
    print(f"\n--- Testing chunk_size={size} ---")
    size_results = []
    for title in contracts_to_test[:2]:  # use 2 contracts to keep runtime reasonable
        size_results.extend(evaluate_contract(title, chunk_size=size, num_questions=3))
    df_size = pd.DataFrame(size_results)
    supported_pct = (df_size["verdict"] == "supported").mean() * 100
    chunk_size_results[size] = supported_pct
    print(f"chunk_size={size}: {supported_pct:.1f}% supported")

print("\nSummary:", chunk_size_results)


## Step 12: Save Results

Download your results table as a CSV for your report.

In [ ]:
results_df.to_csv("trustrag_results.csv", index=False)
print("Saved to trustrag_results.csv")

from google.colab import files
files.download("trustrag_results.csv")
